# Sessions

In [9]:
from dotenv import load_dotenv
import os
from openai import OpenAI
load_dotenv()


client = OpenAI()                      # reads OPENAI_API_KEY from the environment
MODEL = "gpt-4o-mini"

# response = client.chat.completions.create(
#     model=MODEL,
#     messages=[{"role": "user", "content": "Hi, my name is Ramesh and I study transformers."}],
# )
# print(response.choices[0].message.content)


In [10]:
import json
import time
import uuid
from pathlib import Path

In [22]:
SESSIONS_DIR = Path("sessions")
MAX_EVENTS_IN_PROMPT = 5

class Session:
    def __init__(self, user_id: str, session_id: str | None = None):
        self.user_id = user_id                       # WHO the conversation is with
        self.id = session_id or uuid.uuid4().hex[:8]  # WHICH conversation this is
        self.events: list[dict] = []                 # WHAT has been said, in order
        self.state: dict = {}   


    def add_event(self, role: str, content: str) -> None:
        self.events.append({"role": role, "content": content, "time": time.time()})

    def save(self) -> None:
        SESSIONS_DIR.mkdir(exist_ok=True)
        (SESSIONS_DIR / f"{self.id}.json").write_text(json.dumps(self.__dict__, indent=2))

    
    @classmethod
    def load(cls, session_id: str) -> "Session":
        data = json.loads((SESSIONS_DIR / f"{session_id}.json").read_text())
        session = cls(data["user_id"], data["id"])
        session.events, session.state = data["events"], data["state"]
        return session
        
def history_as_messages(session: Session) -> list[dict]:
    """Turns stored events into the message list the model expects (most recent events only)."""
    recent = session.events[-MAX_EVENTS_IN_PROMPT:]
    return [{"role": e["role"], "content": e["content"]} for e in recent]
    


def chat(session: Session, text: str) -> str:
    session.add_event("user", text)
    messages = [{"role": "system", "content": "You are a friendly travel assistant."}]
    messages += history_as_messages(session)

    print("\n\nmessage to LLM:: ", messages, "\n\n")
    reply = client.chat.completions.create(model=MODEL, messages=messages).choices[0].message.content
    session.add_event("assistant", reply)
    session.save()
    return reply


if __name__ == "__main__":
    session = Session(user_id = "Ramesh")

    ai_response = chat(session, "hello my name is Ramesh and i am a software engineer living in mumbai.  i am planning to travel to goa in december!!")
    print("Assistant:  ", ai_response)
    print(f"(saved to sessions/{session.id}.json with {len(session.events)} events)")
    
    # saved_id = session.id
    # del
    



        



message to LLM::  [{'role': 'system', 'content': 'You are a friendly travel assistant.'}, {'role': 'user', 'content': 'hello my name is Ramesh and i am a software engineer living in mumbai.  i am planning to travel to goa in december!!'}] 


Assistant:   Hello Ramesh! That sounds like a fantastic plan! Goa is a beautiful destination with stunning beaches, vibrant nightlife, and a unique blend of Indian and Portuguese cultures. December is a great time to visit due to the pleasant weather and festive atmosphere. 

Do you have any specific places in Goa that you want to visit or activities you’d like to do? I’d be happy to help you with recommendations!
(saved to sessions/2b97db4f.json with 2 events)


In [23]:
ai_response = chat(session, "do you know where i am planning to go now?")
print("Assistant:  ", ai_response)
print(f"(saved to sessions/{session.id}.json with {len(session.events)} events)")



message to LLM::  [{'role': 'system', 'content': 'You are a friendly travel assistant.'}, {'role': 'user', 'content': 'hello my name is Ramesh and i am a software engineer living in mumbai.  i am planning to travel to goa in december!!'}, {'role': 'assistant', 'content': 'Hello Ramesh! That sounds like a fantastic plan! Goa is a beautiful destination with stunning beaches, vibrant nightlife, and a unique blend of Indian and Portuguese cultures. December is a great time to visit due to the pleasant weather and festive atmosphere. \n\nDo you have any specific places in Goa that you want to visit or activities you’d like to do? I’d be happy to help you with recommendations!'}, {'role': 'user', 'content': 'do you know where i am planning to go now?'}] 


Assistant:   I know you're planning to travel to Goa in December, but I don't have details about specific locations or plans beyond that. If you have any particular places in mind, activities you want to do, or if you're looking for reco

In [24]:
ai_response = chat(session, "do you know my name. ")
print("Assistant:  ", ai_response)
print(f"(saved to sessions/{session.id}.json with {len(session.events)} events)")



message to LLM::  [{'role': 'system', 'content': 'You are a friendly travel assistant.'}, {'role': 'user', 'content': 'hello my name is Ramesh and i am a software engineer living in mumbai.  i am planning to travel to goa in december!!'}, {'role': 'assistant', 'content': 'Hello Ramesh! That sounds like a fantastic plan! Goa is a beautiful destination with stunning beaches, vibrant nightlife, and a unique blend of Indian and Portuguese cultures. December is a great time to visit due to the pleasant weather and festive atmosphere. \n\nDo you have any specific places in Goa that you want to visit or activities you’d like to do? I’d be happy to help you with recommendations!'}, {'role': 'user', 'content': 'do you know where i am planning to go now?'}, {'role': 'assistant', 'content': "I know you're planning to travel to Goa in December, but I don't have details about specific locations or plans beyond that. If you have any particular places in mind, activities you want to do, or if you'r

In [ ]:
[
    
    {'role': 'system', 'content': 'You are a friendly travel assistant.'}, 
    
    {'role': 'user', 'content': 'hello my name is Ramesh and i am a software engineer living in mumbai.  i am planning to travel to goa in december!!'}, 
    
    {'role': 'assistant', 'content': 'Hello Ramesh! That sounds like a fantastic plan! Goa is a great destination, especially in December with its pleasant weather and festive vibes. If you need any help with your travel plans—like where to stay, places to visit, or activities to do—feel free to ask!'}, 
    
    {'role': 'user', 'content': 'do you know where i am planning to go now?'}, 
    
    {'role': 'assistant', 'content': "You mentioned that you're planning to travel to Goa in December! If you have specific places or activities in mind within Goa, or if you're looking for recommendations based on your interests, feel free to share!"}, 
    
    {'role': 'user', 'content': 'do you know my name. '}] 

In [25]:
ai_response = chat(session, "do you know my profession. ")
print("Assistant:  ", ai_response)
print(f"(saved to sessions/{session.id}.json with {len(session.events)} events)")



message to LLM::  [{'role': 'system', 'content': 'You are a friendly travel assistant.'}, {'role': 'user', 'content': 'do you know where i am planning to go now?'}, {'role': 'assistant', 'content': "I know you're planning to travel to Goa in December, but I don't have details about specific locations or plans beyond that. If you have any particular places in mind, activities you want to do, or if you're looking for recommendations, feel free to share!"}, {'role': 'user', 'content': 'do you know my name. '}, {'role': 'assistant', 'content': "Yes, your name is Ramesh! If there's anything else you'd like me to know or if you have more questions about your trip to Goa, just let me know!"}, {'role': 'user', 'content': 'do you know my profession. '}] 


Assistant:   I don't have information about your profession. If you'd like to share it or if it relates to your travel plans, feel free to let me know!
(saved to sessions/2b97db4f.json with 8 events)


In [11]:
# messages = [
 
#     {"role": "system", "content": "You are a helpful assistant who is very polite"},
#     {"role": "user", "content": "Hi how are you?"}, 
#     {"role": "assitant", "content": "Hello nice to meet you?"}

    
# ]


# state = {"name" : "ramesh", "city" : "delhi"}

# State

In [29]:
MAX_STEPS = 5


def run_tool(name: str, arguments: str, tool_functions: dict) -> dict:
    """Runs one requested tool. Errors are returned as data the model can read and react to."""
    if name not in tool_functions:
        return {"error": f"Unknown tool {name}."}
    try:
        return tool_functions[name](**json.loads(arguments))
    except Exception as exc:  # bad arguments, a failing service, ...
        return {"error": f"{type(exc).__name__}: {exc}"}

        
def run_agent_loop(messages: list, tool_schemas: list, tool_functions: dict,
                   max_steps: int = MAX_STEPS, verbose: bool = True) -> tuple[str, list]:
    trace = []

    for step in range(1, max_steps + 1):
        # 1. THINK: the model sees the whole conversation and the tool descriptions.
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tool_schemas)
        message = response.choices[0].message
        # print(f"message in step {step}:: {message}")

        # 2. DONE? No tool requested means the model has produced its final answer.
        if not message.tool_calls:
            messages.append({"role": "assistant", "content": message.content})
            return message.content, trace


        # 3. ACT: record the model's tool requests, then run each one.
        messages.append({"role": "assistant", "content": message.content, "tool_calls": [
            {"id": call.id, "type": "function",
             "function": {"name": call.function.name, "arguments": call.function.arguments}}
            for call in message.tool_calls]})

        for call in message.tool_calls:
            result = run_tool(call.function.name, call.function.arguments, tool_functions)
            trace.append({"step": step, "tool": call.function.name,
                          "arguments": call.function.arguments, "result": result})

            if verbose:
                print(f"  step {step}: {call.function.name}({call.function.arguments}) -> {result}")
        
    
            # 4. OBSERVE: the result goes back into the conversation for the next iteration.
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
        
    return "I stopped because I reached the step limit before finishing.", trace
    

In [31]:
import json
from functools import partial
from pathlib import Path

USER_STATE_FILE = Path("user_state.json") 

def load_user_state(user_id: str) -> dict:
    everyone = json.loads(USER_STATE_FILE.read_text()) if USER_STATE_FILE.exists() else {}
    return everyone.get(user_id, {})

def save_user_state(user_id: str, state: dict) -> None:
    everyone = json.loads(USER_STATE_FILE.read_text()) if USER_STATE_FILE.exists() else {}
    everyone[user_id] = state
    USER_STATE_FILE.write_text(json.dumps(everyone, indent=2))


## Save the trip
def set_trip_detail(session: Session, key: str, value: str) -> dict:
    session.state[key] = value                      # session scope
    return {"saved": f"trip {key} = {value}"}

def set_user_preference(session: Session, key: str, value: str) -> dict:
    preferences = load_user_state(session.user_id)  # user scope
    preferences[key] = value
    save_user_state(session.user_id, preferences)
    return {"saved": f"user {key} = {value}"}

STATE_TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": "set_trip_detail",
        "description": "Save a detail of the trip planned in THIS conversation, "
                       "for example destination, nights or dates.",
        "parameters": {"type": "object",
                       "properties": {"key": {"type": "string"}, "value": {"type": "string"}},
                       "required": ["key", "value"]}}},
    {"type": "function", "function": {
        "name": "set_user_preference",
        "description": "Save a lasting preference of the user that applies to ALL future trips, "
                       "for example diet, seat or home city.",
        "parameters": {"type": "object",
                       "properties": {"key": {"type": "string"}, "value": {"type": "string"}},
                       "required": ["key", "value"]}}},
]



def build_system_prompt(session: Session) -> str:
    """READ state: the current state is written into the instructions before every turn."""
    return (
        "You are a travel assistant. Save trip details and lasting user preferences "
        "with your tools as soon as you learn them.\n"
        f"Trip details (this conversation): {json.dumps(session.state)}\n"
        f"User preferences (all conversations): {json.dumps(load_user_state(session.user_id))}"
    )



def chat(session: Session, text: str) -> str:
    session.add_event("user", text)
    messages = [{"role": "system", "content": build_system_prompt(session)}]

    messages += history_as_messages(session)
    
    tools = {"set_trip_detail": partial(set_trip_detail, session),        # bind the tools
             "set_user_preference": partial(set_user_preference, session)}


    answer, _ = run_agent_loop(messages, STATE_TOOL_SCHEMAS, tools)

    session.add_event("assistant", answer)
    session.save()
    return answer

if __name__ == "__main__":
    first = Session(user_id = "Ramesh")
    print("assistant:", chat(first, "I want 3 nights in Goa next month. By the way, I'm vegetarian and i prefer to travel by flights or 1st class AC. i am rich"))
    print("session state:", first.state)
    print("user state:",  load_user_state("Ramesh"))
         



  step 1: set_trip_detail({"key": "destination", "value": "Goa"}) -> {'saved': 'trip destination = Goa'}
  step 1: set_trip_detail({"key": "nights", "value": "3"}) -> {'saved': 'trip nights = 3'}
  step 1: set_user_preference({"key": "diet", "value": "vegetarian"}) -> {'saved': 'user diet = vegetarian'}
  step 1: set_user_preference({"key": "travel_preference", "value": "flights or 1st class AC"}) -> {'saved': 'user travel_preference = flights or 1st class AC'}
  step 1: set_user_preference({"key": "budget", "value": "rich"}) -> {'saved': 'user budget = rich'}
assistant: I've saved your trip details for 3 nights in Goa next month. Additionally, your preferences are noted as vegetarian diet, traveling by flights or 1st class AC, and that you have a rich budget. If you need any more assistance with your trip, feel free to ask!
session state: {'destination': 'Goa', 'nights': '3'}
user state: {'diet': 'vegetarian', 'travel_preference': 'flights or 1st class AC', 'budget': 'rich'}


In [32]:
second = Session(user_id="asha")
print("\nassistant:", chat(second, "Let's plan a weekend in Jaipur. Any food suggestions?"))
print("session state:", second.state)

  step 1: set_trip_detail({"key":"destination","value":"Jaipur"}) -> {'saved': 'trip destination = Jaipur'}

assistant: Jaipur has a rich culinary scene with plenty of delightful options. Here are some food suggestions:

1. **Dal Baati Churma** - A traditional dish of Rajasthani cuisine, it's a must-try.
2. **Gatte Ki Sabzi** - A curry made with gram flour dumplings, best enjoyed with roti or rice.
3. **Ker Sangri** - A unique desert vegetable dish that offers a taste of local flavors.
4. **Laal Maas** - A spicy mutton curry that's famous in Rajasthan.
5. **Sweets** - Don't miss trying local sweets like Ghevar and Ladoo.

You can also explore local street food like Pani Puri, Samosa, and Kachori.

If you have specific preferences or dietary restrictions, let me know, and I can refine these suggestions!
session state: {'destination': 'Jaipur'}


In [33]:
print("\nassistant:", chat(second, "My name is Asha and i am poor and i like to travel by bus and affordable hotels"))

  step 1: set_user_preference({"key": "name", "value": "Asha"}) -> {'saved': 'user name = Asha'}
  step 1: set_user_preference({"key": "budget", "value": "poor"}) -> {'saved': 'user budget = poor'}
  step 1: set_user_preference({"key": "transportation", "value": "bus"}) -> {'saved': 'user transportation = bus'}
  step 1: set_user_preference({"key": "accommodation", "value": "affordable hotels"}) -> {'saved': 'user accommodation = affordable hotels'}

assistant: Nice to meet you, Asha! I’ve noted your preferences: you like to travel by bus, prefer affordable hotels, and are on a budget. If you need more specific recommendations for your weekend in Jaipur or any other help, feel free to ask!
